# F1 Race Analysis

Working notebook — grows alongside the analysis modules.
Each section validates a block against real race data.

**Race used:** 2023 Bahrain GP — clean race, no major weather events, strong teammate battles (VER/PER, HAM/RUS).

Sections:
1. Load & inspect raw lap data
2. Lap time volatility (Block 2)
3. Pairs / cointegration (Block 3)
4. Tyre degradation (Block 5)

In [ ]:
import sys
sys.path.insert(0, '..')  # so imports find data/ and analysis/ from the repo root

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

from data.fastf1_loader import get_accurate_laps, get_teammate_laps

## 1. Load race data

First run will download from the FastF1 API and cache locally — takes ~30s.  
Every run after that is instant from cache.

`get_accurate_laps` returns only `IsAccurate=True` laps — pit in/out laps and SC laps are already stripped.

In [ ]:
YEAR = 2023
EVENT = 'Bahrain'

laps = get_accurate_laps(YEAR, EVENT)

print(f"Total clean laps loaded: {len(laps)}")
print(f"Drivers: {sorted(laps['Driver'].unique())}")
laps.head()

## Lap times across the race — all drivers

Plotting every driver's lap times by lap number.  
What to look for:
- **Stint structure** — clusters of faster laps followed by a gap (pit stop) then faster laps again on fresh tyres
- **Outliers** — any anomalously slow laps that slipped through the `IsAccurate` filter
- **Backmarkers** — should sit visibly above the frontrunners

In [ ]:
drivers = sorted(laps['Driver'].unique())
colors = cm.tab20(np.linspace(0, 1, len(drivers)))

fig, ax = plt.subplots(figsize=(14, 6))

for driver, color in zip(drivers, colors):
    d = laps[laps['Driver'] == driver]
    ax.plot(d['LapNumber'], d['LapTime_s'], label=driver, color=color, linewidth=1, alpha=0.8)

ax.set_xlabel('Lap Number')
ax.set_ylabel('Lap Time (s)')
ax.set_title('2023 Bahrain GP — All drivers lap times')
ax.legend(ncol=4, fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()

## 2. Lap time volatility

Which drivers are most consistent? How does tyre age affect variance?

- `driver_volatility_summary` ranks every driver by lap time std across the whole race — lower std = more consistent
- `stint_volatility` breaks it down per stint so you can see if a driver was erratic on one compound vs another
- Finance parallel: same as ranking stocks by realised volatility over a period, then decomposing by regime

In [ ]:
from analysis.lap_volatility import driver_volatility_summary, stint_volatility

# Overall consistency ranking — who had the most stable lap times across the whole race?
vol_summary = driver_volatility_summary(laps)
print("Driver consistency ranking (lower std = more consistent):")
print(vol_summary.to_string(index=False))

In [ ]:
# Volatility by stint — does a driver get more erratic as tyres wear?
# Each row is one driver's one stint: std tells you how much lap times varied within that stint
stint_vol = stint_volatility(laps)

# Focus on the top 6 drivers for readability
top6 = ['VER', 'PER', 'LEC', 'SAI', 'HAM', 'RUS']
print("Stint volatility — top 6 drivers:")
print(stint_vol[stint_vol['Driver'].isin(top6)].to_string(index=False))

In [ ]:
# Bar chart — driver consistency across the whole race
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(vol_summary['Driver'], vol_summary['std'], color='steelblue', alpha=0.8)
ax.set_xlabel('Driver')
ax.set_ylabel('Lap Time Std Dev (s)')
ax.set_title('2023 Bahrain GP — Driver consistency (lower = more consistent)')
ax.axhline(vol_summary['std'].mean(), color='red', linestyle='--', linewidth=1, label='Field average')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Pairs analysis — VER vs PER

Do Verstappen and Perez's lap times form a mean-reverting spread?

- **Step 1:** OLS regression to find the hedge ratio β (Engle-Granger step 1)
- **Step 2:** ADF test on the OLS residuals — is the spread stationary?
- **Step 3:** Rolling z-score — when was Perez anomalously slow relative to Verstappen?

Finance parallel: exactly what you'd run on PEP vs KO before building a pairs trade.

In [ ]:
from analysis.pairs import pairs_summary, compute_spread, spread_zscore, estimate_hedge_ratio

DRIVER_A = 'VER'
DRIVER_B = 'PER'

# Engle-Granger step 1: OLS hedge ratio
hedge = estimate_hedge_ratio(laps, DRIVER_A, DRIVER_B)
print(f"Hedge ratio (β): {hedge['hedge_ratio']}")
print(f"R²: {hedge['r_squared']}  — how well PER's lap times predict VER's\n")

# Full summary: hedge ratio + ADF on residuals
summary = pairs_summary(laps, DRIVER_A, DRIVER_B)
print("Pairs summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Spread and rolling z-score over the race
spread_df = compute_spread(laps, DRIVER_A, DRIVER_B, hedge_ratio=hedge['hedge_ratio'])
spread_df['zscore'] = spread_zscore(spread_df['spread'], window=10)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Top: raw spread (VER - β*PER in seconds)
ax1.plot(spread_df.index, spread_df['spread'], color='steelblue', linewidth=1.2)
ax1.axhline(spread_df['spread'].mean(), color='red', linestyle='--', linewidth=1, label='Mean')
ax1.set_ylabel('Spread (s)')
ax1.set_title(f'{DRIVER_A} vs {DRIVER_B} — Lap time spread (OLS-adjusted)')
ax1.legend()

# Bottom: z-score — entry/exit signal equivalent
ax2.plot(spread_df.index, spread_df['zscore'], color='darkorange', linewidth=1.2)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axhline(2, color='red', linestyle='--', linewidth=1, label='+2σ')
ax2.axhline(-2, color='green', linestyle='--', linewidth=1, label='−2σ')
ax2.set_ylabel('Z-score')
ax2.set_xlabel('Lap Number')
ax2.set_title('Rolling z-score (window=10) — how far spread deviates from its mean')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Tyre degradation — two-factor OLS

How fast are drivers' tyres actually degrading, and when should they have pitted?

- `deg_summary` fits the two-factor model per stint: `LapTime ~ TyreLife + LapNumber`
  - `deg_rate` (β_tyre): seconds lost per lap of tyre age — the pure rubber wear signal
  - `fuel_effect` (β_fuel): should be negative — car gets lighter and faster as fuel burns off
  - `crossover_lap`: how many laps until accumulated deg cost equals the 22s pit stop time loss
- `field_deg_rates` runs every driver and lets you compare who degraded fastest on the same compound

Finance parallel: deg_rate is the asset's decay rate; crossover_lap is the break-even holding period before the carry cost exceeds the entry cost of switching positions.

In [ ]:
from analysis.tyre_deg import deg_summary, field_deg_rates

# VER degradation profile — one row per stint
ver_deg = deg_summary(laps, 'VER')
print("VER — degradation profile:")
print(ver_deg.to_string(index=False))

print()

# PER for comparison — same team, similar strategy, different driver inputs
per_deg = deg_summary(laps, 'PER')
print("PER — degradation profile:")
print(per_deg.to_string(index=False))

In [ ]:
# Full field — who degraded fastest on each compound?
field = field_deg_rates(laps)
print("Field degradation rates (sorted by compound, then deg_rate ascending):")
print(field[['Driver', 'Stint', 'Compound', 'deg_rate', 'fuel_effect', 'r_squared', 'laps']].to_string(index=False))

In [ ]:
# Bar chart — deg rates by driver on SOFT compound (most laps, clearest signal)
# Crossover lap table for VER and PER underneath

soft_field = field[field['Compound'] == 'SOFT'].copy()

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(soft_field['Driver'], soft_field['deg_rate'], color='tomato', alpha=0.85)
ax.set_xlabel('Driver')
ax.set_ylabel('Deg rate (s / lap of tyre age)')
ax.set_title('2023 Bahrain GP — SOFT compound degradation rate by driver')
ax.axhline(soft_field['deg_rate'].median(), color='navy', linestyle='--', linewidth=1, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

print("\nCrossover laps (at 22s pit loss) — SOFT compound:")
print(f"{'Driver':<8} {'Stint':<7} {'Deg rate':<12} {'Crossover lap'}")
for _, row in soft_field.iterrows():
    cl = row['deg_rate']
    crossover = round(22.0 / cl, 1) if cl > 0 else float('inf')
    print(f"{row['Driver']:<8} {int(row['Stint']):<7} {row['deg_rate']:<12} {crossover}")